# Benchmark image and mask gallery

This notebook builds image plus binary-mask galleries for both benchmarks from the benchmark assets already in the repo.
VGGNet16 is restricted to the image ids present in `results/vggnet16/all_k_vggnet16.csv`.
Flip decisions are read directly from `analysis/swap`; masks render with a white object on black by default, while flipped entries render the object in red on a white background.
Rendered image+mask pairs are also saved into `analysis/images/imagenet` and `analysis/images/cifar100`.

Set `SHOW_ONLY_SWAPPED = True` below if you only want the flipped entries.

In [1]:
from __future__ import annotations

import ast
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'analysis' / 'swap').exists() and (candidate / 'benchmarks').exists():
            return candidate
    raise FileNotFoundError('Could not locate the XAIV repository root from the current working directory.')


def load_assignment_from_notebook(notebook_path: Path, variable_name: str):
    data = json.loads(notebook_path.read_text())
    for cell in data.get('cells', []):
        source = ''.join(cell.get('source', []))
        try:
            tree = ast.parse(source)
        except SyntaxError:
            continue

        for node in tree.body:
            if not isinstance(node, ast.Assign):
                continue
            for target in node.targets:
                if isinstance(target, ast.Name) and target.id == variable_name:
                    return ast.literal_eval(node.value)

    raise KeyError(f'{variable_name!r} was not found in {notebook_path}.')


def load_swap_configuration(root: Path) -> tuple[set[str], set[str]]:
    vgg_selected = load_assignment_from_notebook(
        root / 'analysis' / 'swap' / 'vggnet16.ipynb',
        'SELECTED_IMAGES',
    )
    cifar_targets = load_assignment_from_notebook(
        root / 'analysis' / 'swap' / 'cifar100.ipynb',
        'TARGET_IMAGES',
    )
    cifar_selected = {
        f"label_{item['label']}__idx_{item['idx']}"
        for item in cifar_targets
    }
    return set(vgg_selected), cifar_selected


def load_vgg_result_image_ids(root: Path) -> list[str]:
    results_path = root / 'results' / 'vggnet16' / 'all_k_vggnet16.csv'
    df = pd.read_csv(results_path)
    return sorted(df['image'].dropna().unique().tolist())


def colorize_binary_mask(mask_path: Path, highlight: bool, foreground_value: int = 0) -> np.ndarray:
    mask = np.array(Image.open(mask_path).convert('L'))
    colored = np.zeros((*mask.shape, 3), dtype=np.uint8)
    if highlight:
        colored[:] = 255
    foreground = mask == foreground_value
    foreground_color = np.array([220, 38, 38] if highlight else [255, 255, 255], dtype=np.uint8)
    colored[foreground] = foreground_color
    return colored


def compose_pair_image(
    image_path: Path,
    mask_path: Path,
    highlight: bool,
    gap: int = 8,
    foreground_value: int = 0,
) -> np.ndarray:
    colored_mask = colorize_binary_mask(
        mask_path,
        highlight=highlight,
        foreground_value=foreground_value,
    )
    image_pil = Image.open(image_path).convert('RGB')
    if image_pil.size[::-1] != colored_mask.shape[:2]:
        image_pil = image_pil.resize((colored_mask.shape[1], colored_mask.shape[0]))
    image = np.array(image_pil)
    separator = np.full((image.shape[0], gap, 3), 255, dtype=np.uint8)
    return np.concatenate([image, separator, colored_mask], axis=1)


def build_vgg_items(root: Path, flipped_ids: set[str], allowed_ids: list[str]) -> list[dict[str, object]]:
    benchmark_image_dir = root / 'benchmarks' / 'vggnet16_benchmark2022' / 'imagenet-sample'
    debug_dir = root / 'benchmarks' / 'vggnet16_benchmark2022_segmented_all_imgs' / 'debug_vis'
    items = []
    missing_originals = []
    missing_masks = []

    for image_id in allowed_ids:
        image_path = debug_dir / f'{image_id}_original.png'
        if not image_path.exists():
            image_path = benchmark_image_dir / f'{image_id}.JPEG'
        mask_path = debug_dir / f'{image_id}_seg0_mask_bw.png'
        if not image_path.exists():
            missing_originals.append(image_id)
            continue
        if not mask_path.exists():
            missing_masks.append(image_id)
            continue

        items.append(
            {
                'id': image_id,
                'label': image_id,
                'image_path': image_path,
                'mask_path': mask_path,
                'pair_gap': 8,
                'mask_foreground_value': 0,
                'flipped': image_id in flipped_ids,
            }
        )

    if missing_originals:
        print(f'Skipped {len(missing_originals)} VGG items with missing originals.')
    if missing_masks:
        print(f'Skipped {len(missing_masks)} VGG items with missing masks.')
    return items


def build_cifar_items(root: Path, flipped_ids: set[str]) -> list[dict[str, object]]:
    image_dir = root / 'benchmarks' / 'cifar100_2024' / 'decoded_vnnlibs'
    mask_dir = root / 'benchmarks' / 'cifar100_2024' / 'decoded_vnnlibs_sam2_split' / 'masks'
    items = []
    missing = []
    matched_originals = set()

    for mask_path in sorted(mask_dir.glob('label_*__idx_*.png')):
        base_id = mask_path.stem
        matches = sorted(image_dir.glob(f'*__{base_id}.png'))
        if not matches:
            missing.append(base_id)
            continue

        for image_path in matches:
            matched_originals.add(image_path.stem)
            items.append(
                {
                    'id': image_path.stem,
                    'label': image_path.stem,
                    'image_path': image_path,
                    'mask_path': mask_path,
                    'pair_gap': 2,
                    'mask_foreground_value': 255,
                    'flipped': base_id in flipped_ids,
                }
            )

    if missing:
        print(f'Skipped {len(missing)} CIFAR items with missing originals.')
    originals_without_masks = sorted(
        image_path.stem
        for image_path in image_dir.glob('*.png')
        if image_path.stem not in matched_originals
    )
    if originals_without_masks:
        preview = ', '.join(originals_without_masks[:5])
        print(
            f'CIFAR decoded originals without a matching mask: {len(originals_without_masks)} '
            f'({preview})'
        )
    return items


def save_pair_images(
    items: list[dict[str, object]],
    output_dir: Path,
    title: str,
    *,
    show_only_swapped: bool,
) -> None:
    visible = [item for item in items if item['flipped'] or not show_only_swapped]
    output_dir.mkdir(parents=True, exist_ok=True)

    for item in visible:
        pair_image = compose_pair_image(
            item['image_path'],
            item['mask_path'],
            highlight=bool(item['flipped']),
            gap=int(item.get('pair_gap', 8)),
            foreground_value=int(item.get('mask_foreground_value', 0)),
        )
        Image.fromarray(pair_image).save(output_dir / f"{item['id']}_pair.png")

    print(f'{title}: saved {len(visible)} pair images to {output_dir}')


def chunked(values: list[dict[str, object]], size: int):
    for start in range(0, len(values), size):
        yield values[start : start + size]


def render_gallery(
    items: list[dict[str, object]],
    title: str,
    *,
    show_only_swapped: bool,
    pairs_per_row: int,
    rows_per_figure: int,
) -> None:
    visible = [item for item in items if item['flipped'] or not show_only_swapped]
    if not visible:
        print(f'{title}: nothing to display.')
        return

    items_per_figure = max(1, pairs_per_row * rows_per_figure)
    total_figures = math.ceil(len(visible) / items_per_figure)
    swapped_count = sum(bool(item['flipped']) for item in items)
    print(f'{title}: showing {len(visible)} items ({swapped_count} swapped out of {len(items)} total).')

    for figure_index, batch in enumerate(chunked(visible, items_per_figure), start=1):
        n_rows = math.ceil(len(batch) / pairs_per_row)
        n_cols = pairs_per_row * 2
        fig, axes = plt.subplots(
            n_rows,
            n_cols,
            figsize=(n_cols * 2.15, n_rows * 2.75),
            squeeze=False,
        )

        for ax in axes.ravel():
            ax.axis('off')

        for item_index, item in enumerate(batch):
            row_index, pair_index = divmod(item_index, pairs_per_row)
            image_ax = axes[row_index, pair_index * 2]
            mask_ax = axes[row_index, pair_index * 2 + 1]

            image = np.array(Image.open(item['image_path']).convert('RGB'))
            colored_mask = colorize_binary_mask(
                item['mask_path'],
                highlight=bool(item['flipped']),
                foreground_value=int(item.get('mask_foreground_value', 0)),
            )

            image_ax.imshow(image)
            image_ax.set_title(str(item['label']), fontsize=9, pad=4)
            image_ax.axis('off')

            mask_ax.imshow(colored_mask)
            mask_ax.set_title('mask', fontsize=9, pad=4)
            mask_ax.axis('off')

            if item['flipped']:
                image_ax.text(
                    0.02,
                    0.98,
                    'swapped',
                    transform=image_ax.transAxes,
                    ha='left',
                    va='top',
                    fontsize=8,
                    color='white',
                    bbox={'facecolor': '#dc2626', 'edgecolor': 'none', 'boxstyle': 'round,pad=0.25'},
                )

        mode_label = 'swapped items only' if show_only_swapped else 'full benchmark'
        fig.suptitle(f'{title} ({mode_label}, page {figure_index}/{total_figures})', fontsize=15, y=1.02)
        fig.text(
            0.5,
            0.01,
            'Masks are white-on-black by default; flipped entries render the object red on a white background.',
            ha='center',
            fontsize=10,
        )
        plt.tight_layout(rect=(0, 0.03, 1, 0.98))
        plt.show()


In [2]:
ROOT = find_repo_root()
VGG_FLIPPED, CIFAR_FLIPPED = load_swap_configuration(ROOT)
VGG_RESULT_IMAGE_IDS = load_vgg_result_image_ids(ROOT)

SHOW_ONLY_SWAPPED = False
SAVE_PAIR_IMAGES = True
PAIRS_PER_ROW = 4
ROWS_PER_FIGURE = 4
IMAGENET_PAIR_OUTPUT_DIR = ROOT / 'analysis' / 'images' / 'imagenet'
CIFAR_PAIR_OUTPUT_DIR = ROOT / 'analysis' / 'images' / 'cifar100'

plt.rcParams['figure.dpi'] = 120

vgg_items = build_vgg_items(ROOT, VGG_FLIPPED, VGG_RESULT_IMAGE_IDS)
cifar_items = build_cifar_items(ROOT, CIFAR_FLIPPED)

print(f'Repository root: {ROOT}')
print(f'VGG items: {len(vgg_items)} total from results/vggnet16/all_k_vggnet16.csv, {len(VGG_FLIPPED)} swapped from analysis/swap')
print(f'CIFAR items: {len(cifar_items)} total, {len(CIFAR_FLIPPED)} swapped from analysis/swap')
print(f'ImageNet pair output dir: {IMAGENET_PAIR_OUTPUT_DIR}')
print(f'CIFAR pair output dir: {CIFAR_PAIR_OUTPUT_DIR}')

if SAVE_PAIR_IMAGES:
    save_pair_images(
        vgg_items,
        IMAGENET_PAIR_OUTPUT_DIR,
        'VGGNet16 benchmark',
        show_only_swapped=SHOW_ONLY_SWAPPED,
    )
    save_pair_images(
        cifar_items,
        CIFAR_PAIR_OUTPUT_DIR,
        'CIFAR100 benchmark',
        show_only_swapped=SHOW_ONLY_SWAPPED,
    )

render_gallery(
    vgg_items,
    'VGGNet16 benchmark',
    show_only_swapped=SHOW_ONLY_SWAPPED,
    pairs_per_row=PAIRS_PER_ROW,
    rows_per_figure=ROWS_PER_FIGURE,
)

render_gallery(
    cifar_items,
    'CIFAR100 benchmark',
    show_only_swapped=SHOW_ONLY_SWAPPED,
    pairs_per_row=PAIRS_PER_ROW,
    rows_per_figure=ROWS_PER_FIGURE,
)


CIFAR decoded originals without a matching mask: 1 (resnet_medium__label_43__idx_5506)
Repository root: /Users/zd3504phd/Desktop/XAIV
VGG items: 181 total from results/vggnet16/all_k_vggnet16.csv, 54 swapped from analysis/swap
CIFAR items: 199 total, 2 swapped from analysis/swap
ImageNet pair output dir: /Users/zd3504phd/Desktop/XAIV/analysis/images/imagenet
CIFAR pair output dir: /Users/zd3504phd/Desktop/XAIV/analysis/images/cifar100


VGGNet16 benchmark: saved 181 pair images to /Users/zd3504phd/Desktop/XAIV/analysis/images/imagenet


CIFAR100 benchmark: saved 199 pair images to /Users/zd3504phd/Desktop/XAIV/analysis/images/cifar100
VGGNet16 benchmark: showing 181 items (54 swapped out of 181 total).


/var/folders/fr/3yg6c3f55w58dtlsnjcqjmth0000gq/T/ipykernel_30801/1274568334.py:276: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


CIFAR100 benchmark: showing 199 items (2 swapped out of 199 total).


/var/folders/fr/3yg6c3f55w58dtlsnjcqjmth0000gq/T/ipykernel_30801/1274568334.py:223: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axes = plt.subplots(
